# Visualizing the Crystallizer

The [`Crystallizer`][gemdat.crystallizer.Crystallizer] turns a trajectory's occupancy density
into sites and fits a space group to them (see the [Mobile sites](crystallizer.ipynb)
notebook). This notebook shows two plots for checking such a fit (shown here with plotly; pass
`backend='matplotlib'` for a static matplotlib version):

1. [`plot_sites`][gemdat.plots.crystallizer_sites]: the idealised sites of the chosen space
   group in 3D, coloured by orbit, inside the density they were taken from.
2. [`plot_site_images`][gemdat.plots.crystallizer_site_images]: one site, the images that
   every symmetry operation maps onto it, and the ideal position (their mean), over the
   local density, in three Cartesian views.

Both are methods on the [`CrystallizerScan`][gemdat.crystallizer.CrystallizerScan], and take
`symprec=` to pick a fit other than the best one and `specie=` to show another species than
the mobile one (here Na); the last section shows the S framework.

In [1]:
import plotly.io as pio

# Render plotly figures as static HTML so they display on the docs website
pio.renderers.default = 'plotly_mimetype+notebook_connected'

from gemdat import Trajectory
from gemdat.crystallizer import Crystallizer

# Na3(Sb/W)S4 AIMD run from the `tests/data` submodule (run `git submodule update --init`).
DATA = '../../tests/data/na3sbs4'

trajectory = Trajectory.from_vasprun(
    f'{DATA}/vasprun.xml', cache=f'{DATA}/vasprun.xml.a45e320a.cache'
)
trajectory

Full Formula (Na46 Sb14 W2 S64)
Reduced Formula: Na23Sb7WS32
abc   :  14.437000  14.437028  14.475216
angles:  90.000427  89.999751  90.237694
pbc   :       True       True       True
Constant lattice (True)
Sites (126)
Time steps (25000)

Scan the symmetry tolerance once; every plot below reuses this scan.

In [2]:
crystallizer = Crystallizer.from_trajectory(trajectory, floating_specie='Na', resolution=0.3)
scan = crystallizer.scan(background_level=0.25)
print(scan.format())

symprec (Å)  deviation (Å)  angle dev (°)  space group  #    crystal system  # ops  # orbits  # hits
-----------  -------------  -------------  -----------  ---  --------------  -----  --------  ------
0.01         0              0              P1           1    triclinic       1      128       31    
0.224111     0.09628        0.119          Cc           9    monoclinic      2      64        3     
0.302798     0.152          0.119          Cm           8    monoclinic      4      40        4     
0.452279     0.279          0.238          I-43m        217  cubic           48     10        2     


## Sites in the density

The idealised Na sites of the best fit (I-43m) in 3D, coloured by orbit and labelled with
their Wyckoff letter and multiplicity, inside the Na density. With plotly the density is
drawn as isosurfaces (as in [`plot_3d`][gemdat.plots.plot_3d]) and hovering a site shows its
occupancy; rotate the figure to look along the cell axes.

In [3]:
scan.plot_sites()

## A site and its symmetry images

Every operation of the space group maps one of the sites found in the density (the
watershed sites of this site's orbit) onto this one; those images are the dots, and the
identity contributes the site itself. The ideal position (star) is the mean of the images, up
to an origin shift common to all sites, so their spread is how far this site is from obeying
the symmetry. By default the first Na site is shown; pass `site=` for another and `width=` to
zoom. The background is the density averaged along the viewing direction within the plotted
box.

In [4]:
scan.plot_site_images()

## A framework species: S

The framework species are not taken from density peaks: each framework atom becomes one site
at its time-averaged position. The scan still keeps every species' density
([`scan.density('S')`][gemdat.crystallizer.CrystallizerScan.density]), so the same plots
show how well that single position describes the atom. Here the S density at each site is
a compact blob rather than a smeared rotational shell: the SbS4/WS4 tetrahedra either keep
their orientation or jump between orientations that put the S atoms back on the same
positions, so a single averaged position per atom describes S well.

In [5]:
scan.plot_sites(specie='S')

The S sites are very nearly symmetric: all images of this site lie within a few hundredths
of an Ångstrom of the ideal position, much tighter than for Na.

In [6]:
scan.plot_site_images(specie='S')